In [ ]:
# Sel 1: pasang kebutuhan (Runtime > Change runtime type > T4 GPU)
!pip -q install faster-whisper yt-dlp pyngrok fastapi uvicorn python-multipart nvidia-cublas-cu12 nvidia-cudnn-cu12

In [ ]:
# Sel 2: kode server
"""MAB-Transcribe backend (Google Colab T4): faster-whisper + optional vocal separation + lyric matching + URL download."""
import os, re, uuid, glob, wave, queue, shutil, difflib, tempfile, threading, subprocess
import numpy as np
from fastapi import FastAPI, UploadFile, File, Form, HTTPException
from fastapi.middleware.cors import CORSMiddleware

def _preload_cuda_libs():
    """Colab may ship CUDA 13 while CTranslate2 wants CUDA 12 libs: load the pip 'nvidia-*-cu12' copies first."""
    import ctypes
    for mod, pat in (("nvidia.cublas.lib", "libcublas.so*"), ("nvidia.cudnn.lib", "libcudnn.so*")):
        try:
            pkg = __import__(mod, fromlist=["x"])
            for f in sorted(glob.glob(os.path.join(list(pkg.__path__)[0], pat))):
                ctypes.CDLL(f, mode=ctypes.RTLD_GLOBAL)
        except Exception:
            pass
_preload_cuda_libs()

app = FastAPI(title="MAB-Transcribe")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])
JOBS, Q = {}, queue.Queue()
MODELS = {"turbo": "large-v3-turbo", "large": "large-v3", "small": "small"}
MAX_SECONDS = 15 * 60
_m = {"name": None, "obj": None}

def norm(s):
    return re.sub(r"[\W_]+", " ", s.lower()).split()

def match_lyrics(words, lines):
    """Give each lyric line (correct text) a start time by fuzzy-matching it to the recognised words, in order."""
    out, p = [], 0
    for text in lines:
        t = norm(text); n = len(t)
        if not t:
            continue
        best = (0.0, None, 0)
        for i in range(p, min(len(words), p + 3 * n + 30)):
            for k in range(max(1, n - 2), n + 3):
                seg = [w["t"] for w in words[i:i + k]]
                if not seg:
                    continue
                r = difflib.SequenceMatcher(None, " ".join(t), " ".join(seg)).ratio()
                if r > best[0]:
                    best = (r, i, k)
        if best[1] is not None and best[0] >= 0.35:
            out.append({"text": text, "start": words[best[1]]["s"]}); p = best[1] + best[2]
        else:
            out.append({"text": text, "start": None})
    prev = None                                   # fill unmatched lines by spacing them after the previous one
    for i, o in enumerate(out):
        if o["start"] is None:
            o["start"] = (prev + 2.0) if prev is not None else 0.0
        o["start"] = max(o["start"], prev if prev is not None else 0.0)
        prev = o["start"]
    for i, o in enumerate(out):
        o["end"] = out[i + 1]["start"] if i + 1 < len(out) else o["start"] + 4.0
    return out

def load_model(key):
    name = MODELS.get(key, "large-v3-turbo")
    if _m["name"] != name:
        from faster_whisper import WhisperModel
        import torch
        _m["obj"] = None
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            _m["obj"] = WhisperModel(name, device="cuda", compute_type="float16")
        else:
            _m["obj"] = WhisperModel(name, device="cpu", compute_type="int8")
        _m["name"] = name
    return _m["obj"]

def download(url, d):
    out = os.path.join(d, "dl.%(ext)s")
    r = subprocess.run(["yt-dlp", "-x", "--audio-format", "wav", "--no-playlist", "--match-filter", f"duration<={MAX_SECONDS}", "-o", out, url],
                       capture_output=True, text=True, timeout=600)
    f = glob.glob(os.path.join(d, "dl.*"))
    if r.returncode or not f:
        raise RuntimeError("Gagal mengunduh: platform memblokir, butuh login, atau durasi > 15 menit. " + r.stderr[-150:])
    return f[0]

def separate(path, d):
    """Optional: isolate vocals with Demucs (installed on first use). Returns the vocals file, or None to fall back."""
    try:
        try:
            import demucs  # noqa: F401
        except ImportError:
            subprocess.run(["pip", "install", "-q", "demucs"], check=True, timeout=900)
        subprocess.run(["python", "-m", "demucs.separate", "--two-stems=vocals", "-n", "htdemucs", "--mp3", "-d", "cuda",
                        "-o", os.path.join(d, "sep"), path], check=True, timeout=1800, capture_output=True)
        f = glob.glob(os.path.join(d, "sep", "*", "*", "vocals.mp3"))
        return f[0] if f else None
    except Exception:
        return None

def to_audio(path, d):
    """Decode any audio/video to mono 16 kHz float32 with ffmpeg (avoids PyAV version problems)."""
    out = os.path.join(d, "16k.wav")
    r = subprocess.run(["ffmpeg", "-y", "-i", path, "-vn", "-ac", "1", "-ar", "16000", "-f", "wav", out],
                       capture_output=True, text=True, timeout=600)
    if r.returncode or not os.path.exists(out):
        raise RuntimeError("Gagal membaca audio: " + r.stderr[-150:])
    with wave.open(out, "rb") as w:
        raw = w.readframes(w.getnframes())
    return np.frombuffer(raw, dtype=np.int16).astype(np.float32) / 32768.0

HALLU = {"terima kasih", "terima kasih telah menonton", "terima kasih sudah menonton", "terima kasih banyak", "sampai jumpa",
         "sampai jumpa lagi", "thanks for watching", "thank you", "thank you for watching", "subtitle by the amara org community"}

def keep_seg(s):
    """Drop typical Whisper hallucinations on silence/instrumental parts, and segments it is unsure about."""
    t = " ".join(norm(s.text))
    if not t or t in HALLU:
        return False
    if s.no_speech_prob > 0.6 and s.avg_logprob < -1.0:
        return False
    if s.compression_ratio > 2.4:          # looping/repeated text
        return False
    return True

def run(job):
    j = JOBS[job["id"]]
    try:
        path = job["path"]
        if job["url"]:
            j["message"] = "Mengunduh audio dari URL…"; path = download(job["url"], job["dir"])
        if job["separate"]:
            j["message"] = "Memisahkan vokal… (pertama kali memasang demucs, ±2 menit)"; path = separate(path, job["dir"]) or path
        j["message"] = "Mentranskripsi dengan model " + job["model"] + "…"
        audio = to_audio(path, job["dir"])
        segs, info = load_model(job["model"]).transcribe(
            audio, language=job["lang"] or None, beam_size=5, word_timestamps=True, condition_on_previous_text=False,
            vad_filter=False, initial_prompt=(job["lyrics"][:400] or None))
        segs = [s for s in segs if keep_seg(s)]
        lines = [l.strip() for l in job["lyrics"].splitlines() if l.strip()]
        if lines:
            words = [{"t": x, "s": w.start} for s in segs for w in (s.words or []) for x in norm(w.word)]
            res = match_lyrics(words, lines)
        else:
            res = [{"start": s.start, "end": s.end, "text": s.text.strip()} for s in segs if s.text.strip()]
        j.update(status="done", result={"language": info.language, "lines": res})
    except Exception as e:
        j.update(status="error", message=str(e)[:300])
    finally:
        shutil.rmtree(job["dir"], ignore_errors=True)

def worker():
    while True:
        job = Q.get(); JOBS[job["id"]]["status"] = "running"; run(job)
threading.Thread(target=worker, daemon=True).start()

@app.get("/")
def root():
    try:
        import torch; dev = "cuda" if torch.cuda.is_available() else "cpu"
    except Exception:
        dev = "cpu"
    return {"app": "MAB-Transcribe", "device": dev, "models": list(MODELS)}

@app.post("/transcribe")
async def transcribe(file: UploadFile = File(None), url: str = Form(""), language: str = Form(""), model: str = Form("turbo"),
                     separate: str = Form("0"), lyrics: str = Form("")):
    d, path = tempfile.mkdtemp(), None
    if file is not None:
        path = os.path.join(d, "in.wav")
        with open(path, "wb") as f:
            f.write(await file.read())
    elif not url.strip():
        raise HTTPException(400, "Kirim file atau url")
    jid = uuid.uuid4().hex[:12]
    JOBS[jid] = {"status": "queued", "message": "Antre…"}
    Q.put({"id": jid, "dir": d, "path": path, "url": url.strip(), "lang": language, "model": model, "separate": separate == "1", "lyrics": lyrics})
    return {"job_id": jid}

@app.get("/status/{jid}")
def status(jid: str):
    j = JOBS.get(jid)
    if not j:
        raise HTTPException(404, "Pekerjaan tidak ditemukan")
    return {"status": j["status"], "message": j.get("message", "")}

@app.get("/result/{jid}")
def result(jid: str):
    j = JOBS.get(jid)
    if not j or j["status"] != "done":
        raise HTTPException(404, "Belum selesai")
    return j["result"]


In [ ]:
# Sel 3: jalankan server + ngrok (aman dijalankan berulang). Ganti token di bawah (JANGAN diunggah ke GitHub Public).
NGROK_TOKEN = "PASTE_TOKEN_NGROK_KAMU_DI_SINI"
import threading, socket, uvicorn
from pyngrok import ngrok, conf
conf.get_default().auth_token = NGROK_TOKEN
ngrok.kill()                                    # tutup tunnel lama
s = socket.socket(); busy = s.connect_ex(("127.0.0.1", 8000)) == 0; s.close()
if busy:
    print("Server sudah berjalan, memakai yang lama. (Kalau kode Sel 2 berubah: Runtime > Restart session, lalu jalankan Sel 2 dan 3.)")
else:
    threading.Thread(target=lambda: uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning"), daemon=True).start()
print("Tempel alamat ini di editor (kolom URL backend transkripsi):", ngrok.connect(8000).public_url)